# 問題
学習したロジスティック回帰モデルを用い、検証データの先頭の事例のラベル（ポジネガ）を予測せよ。また、予測されたラベルが検証データで付与されていたラベルと一致しているか、確認せよ。

In [1]:
import spacy
from collections import Counter

# --- pathの準備 ---
path_dev = "./SST-2/dev.tsv"
path_train = "./SST-2/train.tsv"

# --- 形態素解析の準備 ---
nlp = spacy.load("en_core_web_sm")

def integrate_dict(path):
    final_list = []
    with open(path, 'r', encoding='utf-8') as f:
        for row in f:
            row = row.strip().split("\t")
            if not row:
                continue
            if row[1] == "0" or row[1] == "1":
                text = row[0]
                word_counter = Counter()
                doc = nlp(text)
                for tok in doc:
                    # 追加: 軽い前処理（英字のみ＋小文字化）
                    if tok.is_alpha:
                        word_counter.update([tok.text.lower()])
                        tmp_dict = {
                            "text": row[0],
                            "label": row[1],
                            "feature": dict(word_counter),
                        }
                        final_list.append(tmp_dict)
            else:
                print("0と1以外です：", row[1])
    return final_list

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# データ作成
train_items = integrate_dict(path_train)
dev_items   = integrate_dict(path_dev)

X_train_dicts = [d["feature"] for d in train_items]
y_train = np.array([int(d["label"]) for d in train_items])

X_dev_dicts = [d["feature"] for d in dev_items]
y_dev = np.array([int(d["label"]) for d in dev_items])

# 辞書 → 疎行列
vec = DictVectorizer(sparse=True)
X_train = vec.fit_transform(X_train_dicts)
X_dev   = vec.transform(X_dev_dicts)

# 学習（ロジスティック回帰）
clf = LogisticRegression(max_iter=200)
clf.fit(X_train, y_train)

# =========================
# 先頭事例の予測と一致確認
# =========================
label_name = {0: "negative", 1: "positive"}

# 先頭1件を2次元で渡す（形状エラー回避）
pred_label = clf.predict(X_dev[:1])[0]
true_label = y_dev[0]

print("文:", dev_items[0]["text"])
print("予測ラベル:", pred_label, f"({label_name[pred_label]})")
print("正解ラベル:", true_label, f"({label_name[true_label]})")
print("一致している？", pred_label == true_label)

0と1以外です： label
0と1以外です： label
文: it 's a charming and often affecting journey . 
予測ラベル: 1 (positive)
正解ラベル: 1 (positive)
一致している？ True
